# 02 — LLM Insight Generation

Validate the LLM insight generation pipeline end-to-end:
- Load social engagement from `performance.json` and `content_queue.json`
- Page descriptions are fetched live from `registry_clean.json` (sitemap-derived)
- Call `generate_insights()` — the same production function the agent runs on Mondays
- Works with any provider: set `LLM_PROVIDER=ionos` or `LLM_PROVIDER=mistral` in `.env`

**Note:** Umami website analytics are not used here — page URL seeding comes from the sitemap registry.

**To run without S3:** see the optional local storage cell below (Cell 1b).

In [1]:
import sys
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env', override=True)

import os
prefix = os.getenv("S3_STATE_PREFIX", "growth-agent/")
if prefix == "growth-agent/":
    raise RuntimeError(
        f"S3_STATE_PREFIX is {prefix!r} — this is the PRODUCTION prefix. "
        "Set S3_STATE_PREFIX=growth-agent-dev/ in .env before running notebooks."
    )
print(f"Using S3 prefix: {prefix}  ✓")

Using S3 prefix: growth-agent-dev/  ✓


## 1b. Local Storage Mode (optional — skip to use S3)

Run the cell below to use local files instead of S3. Place fixture JSON files in `growth-agent/state/`:

| File | Minimum content |
|---|---|
| `insights.json` | `{}` (or omit — defaults to empty social metrics) |
| `performance.json` | `{"posts": []}` — triggers "no social posts yet" fallback |
| `content_queue.json` | `{"drafts": [], "approved": [], "published": [], "rejected": []}` |
| `registry_clean.json` | `{"urls": ["https://fretchen.eu/quantum/", ...]}` |

`strategy.json` — not needed; code uses hardcoded defaults when absent.

**Tip:** an empty `registry_clean.json` avoids all live HTTP requests to fretchen.eu. Use it to test the engagement block without page description fetching.

In [ ]:
# OPTIONAL: run without S3.
# Run this cell before Cell 2 (Load Data). Skip it to use S3 instead.
import pathlib
from agent.storage import LocalStorage

LOCAL_STATE_DIR = pathlib.Path("../state")
LOCAL_STATE_DIR.mkdir(exist_ok=True)
store = LocalStorage(str(LOCAL_STATE_DIR))
print(f"Using local storage at {LOCAL_STATE_DIR.resolve()}")

In [2]:
import logging

# Surface log output from production code
logging.basicConfig(level=logging.INFO, format='%(name)s %(levelname)s: %(message)s')

# Set provider here — picked up by generate_insights() via os.environ
os.environ['LLM_PROVIDER'] = os.environ.get('LLM_PROVIDER', 'ionos')  # change to 'mistral' if needed

from agent.llm_client import PROVIDERS
provider = os.environ['LLM_PROVIDER']
config = PROVIDERS.get(provider)
key_env = config.api_key_env if config else 'LLM_PROVIDER'
api_key_set = bool(os.environ.get(key_env))

print(f'Active provider : {provider}')
print(f'API key loaded  : {"yes" if api_key_set else f"NO — set {key_env} in .env"}')

/Users/fredjendrzejewski/GitHub/fretchen.github.io/growth-agent/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


Active provider : mistral
API key loaded  : yes


## 2. Load Data

Loads the inputs that `generate_insights()` needs:
- `insights.json` — social metrics (follower counts from ingest node)
- `performance.json` — real Mastodon/Bluesky engagement per post (from ingest node)
- `content_queue.json` — published drafts (used to join post IDs to page URLs)
- `registry_clean.json` — sitemap-derived page URLs (used instead of Umami analytics for page seeding)

If the local storage cell above was run, `store` is already set and the S3 init is skipped.

In [3]:
from agent.models import ContentQueue, Insights, Performance
from agent.storage import S3Storage, load_model

# Create S3 store only if not already set by local mode cell
if 'store' not in dir():
    store = S3Storage(
        bucket=os.getenv('S3_BUCKET'),
        prefix=os.getenv('S3_STATE_PREFIX', 'growth-agent/'),
        access_key=os.getenv('SCW_ACCESS_KEY'),
        secret_key=os.getenv('SCW_SECRET_KEY'),
    )

insights    = load_model(store, 'insights.json',      Insights)
performance = load_model(store, 'performance.json',   Performance)
queue       = load_model(store, 'content_queue.json', ContentQueue)
registry    = store.read('registry_clean.json') or {}

print(f'Social metrics:  {list(insights.social_metrics.keys())}')
print(f'Performance:     {len(performance.posts)} posts with social metrics')
print(f'Queue:           {len(queue.published)} published drafts')
print(f'Registry URLs:   {len(registry.get("urls", []))}')

Social metrics:  ['mastodon', 'bluesky']
Performance:     40 posts with social metrics
Queue:           71 published drafts
Registry URLs:   75


## 2b. Social Engagement Signal

Shows what engagement data the LLM will receive per page URL.
Pages with prior engagement will be classified as `proven`; others as `exploratory`.

In [4]:
from agent.nodes.insights import _build_page_engagement

page_engagement = _build_page_engagement(performance, queue)
if page_engagement:
    for url, e in sorted(page_engagement.items(), key=lambda x: -(x[1]['favourites'] + x[1]['reblogs'])):
        print(f"{url}: {e['favourites']} fav, {e['reblogs']} reb, {e['replies']} rep ({e['posts']} posts)")
else:
    print("No social engagement data — LLM will see '(no social posts yet)'")

https://fretchen.eu/blog/: 2 fav, 0 reb, 1 rep (1 posts)
https://fretchen.eu/quantum/: 2 fav, 0 reb, 0 rep (2 posts)
https://www.fretchen.eu/quantum/amo/15/: 1 fav, 0 reb, 0 rep (1 posts)
https://www.fretchen.eu/quantum/qml/4/: 1 fav, 0 reb, 0 rep (1 posts)
https://www.fretchen.eu/blog/28/: 0 fav, 1 reb, 0 rep (1 posts)
https://www.fretchen.eu/quantum/amo/14/: 1 fav, 0 reb, 0 rep (2 posts)
https://www.fretchen.eu/quantum/qml/1/: 1 fav, 0 reb, 0 rep (1 posts)
https://www.fretchen.eu/lab/: 0 fav, 1 reb, 0 rep (1 posts)
https://www.fretchen.eu/quantum/qml/0/: 1 fav, 0 reb, 0 rep (1 posts)
https://www.fretchen.eu/blog/10/: 0 fav, 0 reb, 0 rep (1 posts)
https://www.fretchen.eu/blog/27/: 0 fav, 0 reb, 0 rep (1 posts)
https://www.fretchen.eu/quantum/basics/2/: 0 fav, 0 reb, 0 rep (2 posts)
https://www.fretchen.eu/imagegen/: 0 fav, 0 reb, 0 rep (1 posts)
https://www.fretchen.eu/blog/7/: 0 fav, 0 reb, 0 rep (1 posts)
https://www.fretchen.eu/blog/4/: 0 fav, 0 reb, 0 rep (1 posts)
https://www.fre

## 3. Run Production Insight Generation

Calls the same `generate_insights()` function the agent uses in production.
The LLM provider and model are picked up from `LLM_PROVIDER` / `LLM_MODEL` env vars.

Expects: ≤5 pages for social, 5 plain-text growth actions (no markdown).
Page descriptions are fetched live from `registry_clean.json` URLs.

In [5]:
from agent.nodes.insights import generate_insights

# Raises on failure — check provider and API key above if this fails
result = generate_insights(store)

print(f'Provider        : {provider}')
print(f'Pages for social: {len(result.best_pages_for_social)}')
print(f'Growth actions  : {len(result.growth_opportunities)}')

httpx INFO: HTTP Request: GET https://www.fretchen.eu/ "HTTP/1.1 200 OK"
httpx INFO: HTTP Request: GET https://www.fretchen.eu/blog/ "HTTP/1.1 200 OK"
httpx INFO: HTTP Request: GET https://www.fretchen.eu/quantum/ "HTTP/1.1 200 OK"
httpx INFO: HTTP Request: GET https://www.fretchen.eu/blog/0/ "HTTP/1.1 200 OK"
httpx INFO: HTTP Request: GET https://www.fretchen.eu/blog/1/ "HTTP/1.1 200 OK"
httpx INFO: HTTP Request: GET https://www.fretchen.eu/blog/10/ "HTTP/1.1 200 OK"
httpx INFO: HTTP Request: GET https://www.fretchen.eu/blog/11/ "HTTP/1.1 200 OK"
httpx INFO: HTTP Request: GET https://www.fretchen.eu/blog/12/ "HTTP/1.1 200 OK"
httpx INFO: HTTP Request: GET https://www.fretchen.eu/blog/13/ "HTTP/1.1 200 OK"
httpx INFO: HTTP Request: GET https://www.fretchen.eu/blog/14/ "HTTP/1.1 200 OK"
httpx INFO: HTTP Request: GET https://www.fretchen.eu/blog/15/ "HTTP/1.1 200 OK"
httpx INFO: HTTP Request: GET https://www.fretchen.eu/blog/16/ "HTTP/1.1 200 OK"
httpx INFO: HTTP Request: GET https://www

Provider        : mistral
Pages for social: 5
Growth actions  : 5


## 4. Inspect Structured Output

In [6]:
print('=== Posts Worth Promoting ===')
for page in result.best_pages_for_social:
    print(f'[{page.selection_type}] {page.url}')
    print(f'  {page.reason}')
    print()

print('=== Growth Actions ===')
for i, opp in enumerate(result.growth_opportunities, 1):
    print(f'{i}. {opp}')

=== Posts Worth Promoting ===
[proven] https://www.fretchen.eu/blog/28/
  This page received a reblog, indicating interest in region-specific economic analysis, which aligns with the political economy focus of the blog. It also stands out as a non-technical piece that could appeal to a broader audience.

[proven] https://www.fretchen.eu/quantum/
  This page has consistent engagement (2 favourites across 2 posts) and covers a niche but highly relevant topic for the target audience. The quantum computing series is a strong candidate for deeper promotion.

[exploratory] https://www.fretchen.eu/blog/13/
  This page combines game theory with interactive content, a unique angle that could drive engagement. While it hasn’t been tested yet, its topic is highly relevant to the blog’s political economy and academic audience.

[exploratory] https://www.fretchen.eu/blog/20/
  This narrative essay explores interdisciplinary themes (climate, democracy, technology) and could resonate with both academ

## 5. Verify S3 Persistence

In [7]:
import json
raw = store.read('llm_analysis.json')
if raw:
    data = raw if isinstance(raw, dict) else json.loads(raw)
    print(f'llm_analysis.json saved: True')
    print(f'Pages for social: {len(data.get("best_pages_for_social", []))}')
    print(f'Growth opportunities: {len(data.get("growth_opportunities", []))}')
else:
    print('llm_analysis.json not found')

llm_analysis.json saved: True
Pages for social: 5
Growth opportunities: 5


In [8]:
print('Done — LLM insight generation validated.')

Done — LLM insight generation validated.
